# Day 07 — OOP Part II: Inheritance + Comprehensions

====================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - Inheritance and super() — extending classes
  - @classmethod and @staticmethod — brief mention
  - Abstract base classes (ABC) — brief mention
  - List comprehensions — filtering and transforming collections
  - Dict comprehensions — building lookup tables

WHY THIS MATTERS FOR LLM ENGINEERING:
  - BaseAgent → ResearchAgent → HITLAgent (Module 05)
  - LangChain's BaseTool, BaseRetriever, BaseMemory are all inherited
  - Comprehensions are how you process 100 search results in one line


In [ ]:

import logging
from abc import ABC, abstractmethod   # mentioned briefly — used by LangChain
from typing import Any

log = logging.getLogger(__name__)




## ABSTRACT BASE CLASS — brief introduction

═════════════════════════════════════════════════════════════════════════════
ABC (Abstract Base Class) forces subclasses to implement specific methods.
If a subclass does NOT implement an @abstractmethod, Python raises TypeError
when you try to create an instance of that subclass.
You will see this pattern everywhere in LangChain:
class BaseTool(ABC):
@abstractmethod
def _run(self, query: str) -> str: ...
You DON'T need to write your own ABCs in Module 01.
But you need to RECOGNISE this pattern when you see it.


In [ ]:

class BaseLLMClient(ABC):
    """
    Abstract base class for LLM client wrappers.

    Any class that inherits from BaseLLMClient MUST implement:
    - call() — send a prompt, get a response
    - get_model_name() — return the model identifier

    This is the same pattern as LangChain's BaseLanguageModel.
    """

    def __init__(self, max_tokens: int = 512, temperature: float = 0.2) -> None:
        self.max_tokens  = max_tokens
        self.temperature = temperature

    @abstractmethod
    def call(self, prompt: str) -> str:
        """Send a prompt and return the LLM response. Must be implemented."""
        ...   # Abstract methods have no body (use ... or pass)

    @abstractmethod
    def get_model_name(self) -> str:
        """Return the model identifier string. Must be implemented."""
        ...

    # Concrete methods can live in the abstract base class.
    # Subclasses inherit these for free (no need to re-implement).
    def count_tokens_estimate(self, text: str) -> int:
        """
        Rough token count estimate (1 token ≈ 4 characters for English text).

        This is a shared utility — all LLM clients can use this without
        implementing it themselves.
        """
        return len(text) // 4

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"model={self.get_model_name()!r}, "
            f"max_tokens={self.max_tokens}, "
            f"temp={self.temperature}"
            f")"
        )




## CONCRETE CLASS: OpenAIClient — inherits from BaseLLMClient

═════════════════════════════════════════════════════════════════════════════


In [ ]:

class OpenAIClient(BaseLLMClient):
    """
    LLM client for OpenAI models (gpt-4o, gpt-4-turbo, etc.)

    Inherits from BaseLLMClient and implements the two abstract methods.
    Also adds OpenAI-specific configuration (api_key, base_url).
    """

    # Class variable — default model for all OpenAIClient instances
    DEFAULT_MODEL = "gpt-4o"

    def __init__(
        self,
        api_key   : str,
        model     : str = DEFAULT_MODEL,
        max_tokens: int = 512,
        temperature: float = 0.2,
    ) -> None:
        # super().__init__() calls the parent class __init__
        # ALWAYS call super().__init__() when overriding __init__
        # so the parent's setup code runs too.
        super().__init__(max_tokens=max_tokens, temperature=temperature)

        # OpenAI-specific attributes (not in the base class)
        self._api_key = api_key   # _api_key: private by convention, don't expose
        self.model    = model

    def call(self, prompt: str) -> str:
        """
        Send a prompt to the OpenAI API and return the response.

        This implements the abstract method from BaseLLMClient.
        In a real implementation this would use the `openai` package.
        Here we simulate the call.
        """
        token_estimate = self.count_tokens_estimate(prompt)  # inherited from base
        log.info(f"OpenAI call | model={self.model} | ~{token_estimate} tokens")

        # Simulated response (real code: openai.chat.completions.create())
        return f"[OpenAI {self.model}] Response to: {prompt[:50]}..."

    def get_model_name(self) -> str:
        """Return the model identifier. Implements the abstract method."""
        return self.model




## SUBCLASS: ResearchAgent — specialises OpenAIClient with search capability

═════════════════════════════════════════════════════════════════════════════


In [ ]:

class ResearchAgent(OpenAIClient):
    """
    An LLM agent that can search a knowledge base before calling the LLM.

    Inherits from OpenAIClient and overrides call() to add:
    1. Search the product knowledge base
    2. Inject the retrieved context into the prompt
    3. Call the LLM with the enriched prompt

    This is a simplified version of the RAG pattern taught in Module 04.
    """

    def __init__(
        self,
        api_key       : str,
        knowledge_base: list[dict],   # list of document dicts to search
        model         : str = OpenAIClient.DEFAULT_MODEL,
    ) -> None:
        # Call the OpenAIClient __init__ via super()
        super().__init__(api_key=api_key, model=model)

        # ResearchAgent-specific attribute
        self._knowledge_base = knowledge_base

    def _search(self, query: str, top_k: int = 3) -> list[dict]:
        """
        Simple keyword search over the knowledge base.

        In Module 04 this becomes a vector similarity search (RAG).
        For now: exact keyword match.

        Args:
            query: The user query to search for.
            top_k: Maximum number of results to return.

        Returns:
            List of matching document dicts.
        """

        query_lower = query.lower()

        # List comprehension with filter condition — much cleaner than a for loop
        # [item for item in collection if condition]
        matches = [
            doc for doc in self._knowledge_base
            if any(
                word in doc.get("content", "").lower()
                for word in query_lower.split()
            )
        ]

        return matches[:top_k]

    def call(self, prompt: str) -> str:
        """
        Override OpenAIClient.call() to add RAG-style context injection.

        Method resolution order (MRO): Python calls ResearchAgent.call()
        first. If this method did not exist, Python would call
        OpenAIClient.call(), then BaseLLMClient (but it's abstract).
        """

        # Step 1: search the knowledge base
        docs = self._search(prompt)

        if docs:
            # Step 2: build context from search results
            context = "\n".join(
                f"[Doc {i+1}] {doc.get('content', '')[:200]}"
                for i, doc in enumerate(docs)
            )

            # Step 3: inject context into the prompt (RAG pattern)
            enriched_prompt = (
                f"<context>\n{context}\n</context>\n\n"
                f"<question>{prompt}</question>"
            )

            log.info(f"ResearchAgent: injected {len(docs)} context docs")
        else:
            enriched_prompt = prompt
            log.warning("ResearchAgent: no relevant docs found for query")

        # Step 4: call the parent's LLM call with the enriched prompt
        # super().call() explicitly calls OpenAIClient.call()
        return super().call(enriched_prompt)

    def __repr__(self) -> str:
        return (
            f"ResearchAgent("
            f"model={self.model!r}, "
            f"kb_size={len(self._knowledge_base)} docs"
            f")"
        )




## @classmethod and @staticmethod — BRIEF MENTION

═════════════════════════════════════════════════════════════════════════════
You will see @classmethod and @staticmethod in LangChain source code.
Knowing what they are prevents confusion.
@classmethod uses `cls` (the CLASS) instead of `self` (the instance).
→ Used for alternative constructors ("factory methods")
→ cls.from_csv_row() in Day 06's ProductReview is a @classmethod
@staticmethod has neither `cls` nor `self`.
→ Used for utility functions that logically belong in a class
but don't need access to the class or instance
Neither is commonly needed when writing your own agents in Module 01-06.


In [ ]:

class PromptVersionManager:
    """Manages prompt versions — brief @classmethod/@staticmethod demo."""

    _versions: list[dict] = []

    @classmethod
    def register(cls, version: int, description: str) -> None:
        """Register a new prompt version. Uses cls to access class state."""
        cls._versions.append({"version": version, "description": description})

    @staticmethod
    def format_version_tag(version: int) -> str:
        """Format a version number as 'v001' style. No class/instance needed."""
        return f"v{version:03d}"   # zero-pad to 3 digits

    @classmethod
    def latest(cls) -> dict | None:
        """Return the most recently registered version."""
        return cls._versions[-1] if cls._versions else None




## SECTION 4: LIST AND DICT COMPREHENSIONS

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def filter_high_rated_products(
    products: list[dict],
    min_price: float = 0.0,
    min_stock: int = 1,
) -> list[dict]:
    """
    Filter products using a list comprehension.

    List comprehension syntax:
        [expression for item in iterable if condition]

    Equivalent for loop (more verbose):
        result = []
        for product in products:
            if condition:
                result.append(product)
        return result

    Args:
        products : List of product dicts from products.csv.
        min_price: Minimum price to include.
        min_stock: Minimum stock quantity to include.

    Returns:
        Filtered list of product dicts.
    """

    # Multiple conditions in one comprehension (and logic)
    return [
        product for product in products
        if float(product.get("price", 0)) >= min_price
        and int(product.get("stock_quantity", 0)) >= min_stock
    ]



In [ ]:
def build_product_index(products: list[dict]) -> dict[int, dict]:
    """
    Build a product lookup dict from a list of product dicts.

    Dict comprehension syntax:
        {key_expr: value_expr for item in iterable if condition}

    Result: {2001: {...product dict...}, 2002: {...}, ...}

    This index makes product lookup O(1) instead of O(n).
    When an LLM calls the lookup_product tool with a product_id,
    you do: product_index[product_id] instead of searching the list.

    Args:
        products: List of product dicts.

    Returns:
        Dict mapping product_id (int) to product dict.
    """

    return {
        int(p["product_id"]): p
        for p in products
        if "product_id" in p   # skip any rows with missing product_id
    }



In [ ]:
def extract_unique_categories(products: list[dict]) -> list[str]:
    """
    Extract unique category names using a set comprehension, then sort.

    Set comprehension: {expression for item in iterable}
    Sets are unordered and deduplicated automatically.

    Args:
        products: List of product dicts.

    Returns:
        Sorted list of unique category name strings.
    """

    # Set comprehension — curly braces, like dict but without the colon
    categories: set[str] = {
        p["category"].strip()
        for p in products
        if "category" in p and p["category"]
    }

    # Convert set to sorted list for consistent ordering
    return sorted(categories)



In [ ]:
def format_products_for_prompt(products: list[dict], max_items: int = 5) -> str:
    """
    Format product data for injection into an LLM prompt.

    Combines slicing, list comprehension, and string joining.

    Args:
        products : List of product dicts.
        max_items: Maximum products to include (avoids context overflow).

    Returns:
        A formatted multi-line string ready for prompt injection.
    """

    # Slice to max_items first, then comprehend (efficient — no unnecessary work)
    lines = [
        f"- {p.get('product_name', 'Unknown'):30s} | "
        f"${float(p.get('price', 0)):.2f} | "
        f"Stock: {p.get('stock_quantity', 0)}"
        for p in products[:max_items]
    ]

    return "\n".join(lines)




In [ ]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

    print("=" * 60)
    print("DAY 07 — OOP II: Inheritance + Comprehensions")
    print("=" * 60)

    # Inheritance
    print("\n[1] Agent class hierarchy")
    knowledge_base = [
        {"content": "Classic Monitor is a 27-inch display with 4K resolution priced at $205.21"},
        {"content": "Ultimate Perfume is a luxury fragrance product in the Beauty category"},
        {"content": "ShopSmart return policy: 30 days, original packaging required"},
    ]

    client = OpenAIClient(api_key="sk-mock", model="gpt-4o")
    agent  = ResearchAgent(api_key="sk-mock", knowledge_base=knowledge_base)

    print(f"  OpenAIClient  : {client}")
    print(f"  ResearchAgent : {agent}")
    print(f"\n  OpenAIClient.call():")
    print(f"    {client.call('Tell me about the Classic Monitor')}")
    print(f"\n  ResearchAgent.call() (with RAG):")
    print(f"    {agent.call('What is the price of Classic Monitor?')}")

    # isinstance with inheritance
    print("\n[2] isinstance() respects inheritance")
    print(f"  isinstance(agent, ResearchAgent) : {isinstance(agent, ResearchAgent)}")
    print(f"  isinstance(agent, OpenAIClient)  : {isinstance(agent, OpenAIClient)}")
    print(f"  isinstance(agent, BaseLLMClient) : {isinstance(agent, BaseLLMClient)}")

    # @classmethod and @staticmethod
    print("\n[3] @classmethod and @staticmethod")
    PromptVersionManager.register(1, "Initial prompt")
    PromptVersionManager.register(2, "Added few-shot examples")
    print(f"  Latest version  : {PromptVersionManager.latest()}")
    print(f"  Formatted tag   : {PromptVersionManager.format_version_tag(2)}")

    # Comprehensions
    print("\n[4] List + Dict Comprehensions")
    sample_products = [
        {"product_id": "2001", "product_name": "Classic Monitor",   "category": "Electronics",  "price": "205.21", "stock_quantity": "238"},
        {"product_id": "2002", "product_name": "Ultimate Perfume",  "category": "Beauty",       "price": "568.17", "stock_quantity": "10"},
        {"product_id": "2003", "product_name": "Budget Headphones", "category": "Electronics",  "price": "29.99",  "stock_quantity": "0"},
        {"product_id": "2004", "product_name": "Yoga Mat",          "category": "Sports",       "price": "45.00",  "stock_quantity": "150"},
    ]

    in_stock = filter_high_rated_products(sample_products, min_price=30.0, min_stock=1)
    print(f"  In-stock products ≥ $30: {[p['product_name'] for p in in_stock]}")

    index = build_product_index(sample_products)
    print(f"  Product index keys: {list(index.keys())}")
    print(f"  Lookup product 2002: {index[2002]['product_name']}")

    categories = extract_unique_categories(sample_products)
    print(f"  Unique categories: {categories}")

    print(f"\n  Formatted for prompt:\n{format_products_for_prompt(sample_products)}")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day07_oop_inheritance.py', run_name='__main__')
